Notebook 1: Configuração do ambiente e carregamento do dataset

In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

arquivo_nome = "ar-condicionado.csv"

diretorio_atual = Path.cwd()

caminhos_encontrados = list(diretorio_atual.rglob(arquivo_nome))

if not caminhos_encontrados:
    caminhos_encontrados = list(diretorio_atual.parent.rglob(arquivo_nome))

if caminhos_encontrados:
    caminho_final = caminhos_encontrados[0]
    print(f"Arquivo localizado com sucesso em: {caminho_final}")
    
    df = pd.read_csv(caminho_final)

    df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('(', '').str.replace(')', '').str.replace('°C', 'C').str.replace('/', '_')
    
    print("Dataset carregado e colunas padronizadas!")
else:
    caminho_manual = Path("../Datasets Itens de casa/ar-condicionado.csv")
    df = pd.read_csv(caminho_manual)
    df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('(', '').str.replace(')', '').str.replace('°C', 'C').str.replace('/', '_')
    print("Carregado via caminho manual!")

Arquivo localizado com sucesso em: c:\Users\mathe\OneDrive\Área de Trabalho\Projeto TCS\Datasets Items de casa\ar-condicionado.csv
Dataset carregado e colunas padronizadas!


Notebook 2: Seleção de atributos (Feature selection)

In [9]:
colunas_para_remover = [
    'Se_outro_fluido_refrigerante,_especificar',
    'Código_de_Barras',
    'Status_do_Modelo',
    'Status_do_Registro',
    'Número_do_Registro'
]

colunas_existentes = [col for col in colunas_para_remover if col in df.columns]
df = df.drop(columns=colunas_existentes)

print(f"✓ {len(colunas_existentes)} colunas irrelevantes removidas.")
print(f"Colunas restantes ({len(df.columns)}):", list(df.columns))

✓ 5 colunas irrelevantes removidas.
Colunas restantes (24): ['Marca', 'Modelo', 'Tipo', 'Modelo_Unidade_Interna', 'Modelo_Unidade_Externa', 'Função', 'Tensão_V', 'Rotação', 'Fluido_Refrigerante', 'Carga_do_Fluido_Refrigerante_g', 'Capacidade_de_Refrigeração_Nominal_Btu_h', 'Capacidade_de_Refrigeração_Nominal_W', 'Potência_Consumida_no_Modo_Espera_W', 'Capacidade_a_35_C_e_em_Carga_Total_W', 'Potência_Consumida_a_35_C_e_em_Carga_Total_W', 'Capacidade_a_35_C_e_em_Carga_Parcial_W', 'Potência_Consumida_a_35_C_e_em_Carga_Parcial_W', 'Capacidade_a_29_C_e_em_Carga_Parcial_W', 'Potência_Consumida_a_29_C_e_em_Carga_Parcial_W', 'Capacidade_a_29_ºC_e_Carga_Total_W', 'Potência_Consumida_a_29_ºC_e_Carga_Total_W', 'IDRS_Wh_Wh', 'Consumo_Anual_de_Energia_kWh', 'Faixa_de_Classificação']


Notebook 3: Tratamento de Duplicatas e Registros Redundantes

In [10]:
linhas_antes = len(df)
df = df.drop_duplicates()
print(f"✓ Linhas duplicadas exatas removidas: {linhas_antes - len(df)}")

if 'Modelo' in df.columns:
    df['contagem_nulos'] = df.isnull().sum(axis=1)
    df = df.sort_values('contagem_nulos').drop_duplicates(subset=['Modelo'], keep='first')
    df = df.drop(columns=['contagem_nulos'])
    print(f"✓ Modelos duplicados tratados. Total de linhas atual: {len(df)}")

✓ Linhas duplicadas exatas removidas: 357
✓ Modelos duplicados tratados. Total de linhas atual: 6589


Notebook 4: Filtragem de Erros e Outliers Irrealistas.

In [11]:
if 'Consumo_Anual_de_Energia_kWh' in df.columns:
    df = df[df['Consumo_Anual_de_Energia_kWh'] > 50]

coluna_potencia = 'Potência_Consumida_a_35_C_e_em_Carga_Total_W'
if coluna_potencia in df.columns:
    df = df[df[coluna_potencia] < 4000]

print(f"✓ Filtros de inconsistência aplicados. Linhas restantes: {len(df)}")

✓ Filtros de inconsistência aplicados. Linhas restantes: 2296


Notebook 5: Imputsação Inteligente e Exportação dos Dados Limpos.

In [12]:
colunas_imputar = [
    'Potência_Consumida_a_35_C_e_em_Carga_Total_W',
    'Consumo_Anual_de_Energia_kWh'
]

colunas_validas = [c for c in colunas_imputar if c in df.columns]

if 'Marca' in df.columns and 'Capacidade_de_Refrigeração_Nominal_Btu_h' in df.columns:
    df[colunas_validas] = df.groupby(['Marca', 'Capacidade_de_Refrigeração_Nominal_Btu_h'])[colunas_validas].transform(lambda x: x.fillna(x.mean()))

df = df.dropna(subset=colunas_validas)

print(f"✓ Processo finalizado! Linhas prontas para uso: {len(df)}")

caminho_saida = caminho_final.parent / "ar-condicionado_limpo.csv"
df.to_csv(caminho_saida, index=False)
print(f"✓ Dataset limpo salvo com sucesso em: {caminho_saida}")

✓ Processo finalizado! Linhas prontas para uso: 2296
✓ Dataset limpo salvo com sucesso em: c:\Users\mathe\OneDrive\Área de Trabalho\Projeto TCS\Datasets Items de casa\ar-condicionado_limpo.csv
